# 01 — Data Cleaning (DeliveryETA)

Ce notebook reprend **étape par étape** les actions du script `src/preprocessing/cleaning.py` (classe `DataCleaner`), avec pour chaque étape :

- **Action** : ce que fait le code
- **Justification** : pourquoi on le fait, et ce que montrent les résultats

**Ordre du pipeline :** chargement → inspection → nettoyage du texte → conversion des types → valeurs manquantes → outliers → doublons → validation → sauvegarde.

## 0. Imports et chemins

In [40]:
import pandas as pd

RAW_PATH = "../data/raw/dataset.csv"
CLEAN_PATH = "../data/processed/dataset_delivery_time_clean.csv"

## 1. Chargement du dataset

**Action :** lecture du CSV brut avec `pd.read_csv`.


In [42]:
df = pd.read_csv(RAW_PATH)
df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


## 2. STRUCTURE DU DATASET

**Action :** Aficher le structure du dataset


In [43]:
print("Dimensions :", df.shape)

print("\nColonnes :")
print(df.columns.tolist())

print("\nTypes et valeurs non nulles :")
df.info()

print("\nStatistiques numériques :")
print(df.describe())

Dimensions : (45593, 20)

Colonnes :
['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)']

Types et valeurs non nulles :
<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  str    
 1   Delivery_person_ID           45593 non-null  str    
 2   Delivery_person_Age          45593 non-null  str    
 3   Delivery_person_Ratings      45593 non-null  str    
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude  

drop unused columns 

In [61]:
df = df.drop(columns=[
    "Delivery_person_ID",
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Order_Date",
    "Time_Orderd",
    "Vehicle_condition",
    "Type_of_order",
    "multiple_deliveries",
    "Festival",
    "City",
])


KeyError: "['Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Order_Date', 'Time_Orderd', 'Vehicle_condition', 'Type_of_order', 'multiple_deliveries', 'Festival', 'City'] not found in axis"

## 3. VALEURS MANQUANTES 
**Action :** afficher les valeurs manquantes


In [45]:
missing = df.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False))

Series([], dtype: int64)


## 4 OUTLIERS
**Action :** aficher le numbre outliers in all column nimerique

## 5. DOUBLONS

**Action :** aficher le number DOUBLONS

In [46]:
duplicates = df.duplicated().sum()

print("\n DOUBLONS ")
print("Nombre de doublons :", duplicates)


 DOUBLONS 
Nombre de doublons : 0


categorical values

In [47]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n===== {col} =====")
    print(df[col].value_counts(dropna=False).head(20))


===== ID =====
ID
0x4607     1
0xb379     1
0x5d6d     1
0x7a6a     1
0x70a2     1
0x9bb4     1
0x95b4     1
0x9eb2     1
0x1102     1
0xcdcd     1
0xd987     1
0x2784     1
0xc8b6     1
0xdb64     1
0x3af3     1
0x3aab     1
0x689b     1
0x6f67     1
0xc9cf     1
0x36b8     1
Name: count, dtype: int64

===== Time_Order_picked =====
Time_Order_picked
21:30:00    496
22:50:00    474
22:40:00    458
18:40:00    457
21:45:00    456
17:55:00    456
22:25:00    455
18:05:00    454
20:50:00    453
23:50:00    453
20:00:00    452
19:55:00    447
18:45:00    447
23:05:00    446
21:55:00    444
23:35:00    443
18:25:00    443
19:40:00    442
20:40:00    440
21:50:00    440
Name: count, dtype: int64

===== Weatherconditions =====
Weatherconditions
conditions Fog           7654
conditions Stormy        7586
conditions Cloudy        7536
conditions Sandstorms    7495
conditions Windy         7422
conditions Sunny         7284
conditions NaN            616
Name: count, dtype: int64

===== Road_tra

/tmp/ipykernel_364561/731942264.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


Remove WHITE SPACE from columns

In [48]:
df.columns = df.columns.str.strip()

for col in df.select_dtypes("object"):
    df[col] = df[col].str.strip()

df = df.replace(["NaN", "nan", ""], pd.NA)

/tmp/ipykernel_364561/3335555437.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes("object"):


handle Weatherconditions  column

In [49]:
df["Weatherconditions"] = (
    df["Weatherconditions"]
    .str.replace("conditions ", "", regex=False)
    .str.strip()
)

In [50]:
print(df["Weatherconditions"].value_counts(dropna=False))

Weatherconditions
Fog           7654
Stormy        7586
Cloudy        7536
Sandstorms    7495
Windy         7422
Sunny         7284
NaN            616
Name: count, dtype: int64


In [51]:
df.isna().sum().sort_values(ascending=False)

Road_traffic_density           601
ID                               0
Restaurant_latitude              0
Restaurant_longitude             0
Delivery_location_longitude      0
Delivery_location_latitude       0
Time_Order_picked                0
Weatherconditions                0
Type_of_vehicle                  0
Time_taken(min)                  0
dtype: int64

fix the Time_Taken column 

In [52]:
df["Time_taken(min)"] = (
    df["Time_taken(min)"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")[0]
)

df["Time_taken(min)"] = pd.to_numeric(
    df["Time_taken(min)"],
    errors="coerce"
)
print(df["Time_taken(min)"].head(10))
print(df["Time_taken(min)"].dtype)
print(df["Time_taken(min)"].isna().sum())

0    24
1    33
2    26
3    21
4    30
5    26
6    40
7    32
8    34
9    46
Name: Time_taken(min), dtype: int64
int64
0


fix time oredred to date format

In [55]:


df["Time_Order_picked"] = pd.to_datetime(
    df["Time_Order_picked"],
    format="%H:%M:%S",
    errors="coerce"
)


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Restaurant_latitude          45593 non-null  float64       
 1   Restaurant_longitude         45593 non-null  float64       
 2   Delivery_location_latitude   45593 non-null  float64       
 3   Delivery_location_longitude  45593 non-null  float64       
 4   Time_Order_picked            45593 non-null  datetime64[us]
 5   Weatherconditions            45593 non-null  str           
 6   Road_traffic_density         44992 non-null  str           
 7   Type_of_vehicle              45593 non-null  str           
 8   Time_taken(min)              0 non-null      Float64       
dtypes: Float64(1), datetime64[us](1), float64(4), str(3)
memory usage: 4.5 MB


check invalid coordinates

In [33]:
coord_cols = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

zero_coords = (
    (df["Restaurant_latitude"] == 0) |
    (df["Restaurant_longitude"] == 0) |
    (df["Delivery_location_latitude"] == 0) |
    (df["Delivery_location_longitude"] == 0)
)

print("Nombre de lignes avec au moins une coordonnée = 0 :", zero_coords.sum())

Nombre de lignes avec au moins une coordonnée = 0 : 3640


In [34]:
for col in coord_cols:
    print(f"\n===== {col} =====")
    print("Min :", df[col].min())
    print("Max :", df[col].max())
    print("Negative :", (df[col] < 0).sum())
    print("Zero :", (df[col] == 0).sum())


===== Restaurant_latitude =====
Min : -30.905562
Max : 30.914057
Negative : 431
Zero : 3640

===== Restaurant_longitude =====
Min : -88.366217
Max : 88.433452
Negative : 162
Zero : 3640

===== Delivery_location_latitude =====
Min : 0.01
Max : 31.054057
Negative : 0
Zero : 0

===== Delivery_location_longitude =====
Min : 0.01
Max : 88.563452
Negative : 0
Zero : 0


In [36]:
print("Nombre total de lignes :", len(df))
print("Lignes avec au moins un 0 :", zero_coords.sum())
print("Pourcentage :", round(zero_coords.mean() * 100, 2), "%")

Nombre total de lignes : 45593
Lignes avec au moins un 0 : 3640
Pourcentage : 7.98 %


In [62]:
coord_cols = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

zero_coords = (df[coord_cols] == 0).any(axis=1)

df = df.loc[~zero_coords].copy()

print("Après :", df.shape)
print("Lignes supprimées :", zero_coords.sum())

Après : (41953, 10)
Lignes supprimées : 3640


In [38]:
coord_cols = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

for col in coord_cols:
    print(f"\n===== {col} =====")
    print("Valeurs < 1 :", (df[col].abs() < 1).sum())
    print("Valeurs < 5 :", (df[col].abs() < 5).sum())
    print("Minimum :", df[col].min())


===== Restaurant_latitude =====
Valeurs < 1 : 0
Valeurs < 5 : 0
Minimum : -30.905562

===== Restaurant_longitude =====
Valeurs < 1 : 0
Valeurs < 5 : 0
Minimum : -88.366217

===== Delivery_location_latitude =====
Valeurs < 1 : 0
Valeurs < 5 : 0
Minimum : 9.967144

===== Delivery_location_longitude =====
Valeurs < 1 : 0
Valeurs < 5 : 0
Minimum : 72.778726


replace categorical data with mode

In [ ]:
categorical_cols = [
    "Weatherconditions",
    "Road_traffic_density",
    "Type_of_vehicle"
]

numeric_cols = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])



    

<class 'pandas.DataFrame'>
Index: 41953 entries, 0 to 45592
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   ID                           41953 non-null  str           
 1   Restaurant_latitude          41953 non-null  float64       
 2   Restaurant_longitude         41953 non-null  float64       
 3   Delivery_location_latitude   41953 non-null  float64       
 4   Delivery_location_longitude  41953 non-null  float64       
 5   Time_Order_picked            41953 non-null  datetime64[us]
 6   Weatherconditions            41953 non-null  str           
 7   Road_traffic_density         41953 non-null  str           
 8   Type_of_vehicle              41953 non-null  str           
 9   Time_taken(min)              41953 non-null  int64         
dtypes: datetime64[us](1), float64(4), int64(1), str(4)
memory usage: 4.5 MB
<class 'pandas.DataFrame'>
Index: 41953 entries, 0

determine outliers

In [68]:
numeric_cols = df.select_dtypes(include="number").columns

print(numeric_cols)

Index(['Restaurant_latitude', 'Restaurant_longitude',
       'Delivery_location_latitude', 'Delivery_location_longitude',
       'Time_taken(min)'],
      dtype='str')


In [69]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    
    print(f"\n===== {col} =====")
    print(f"Q1       : {Q1:.2f}")
    print(f"Q3       : {Q3:.2f}")
    print(f"IQR      : {IQR:.2f}")
    print(f"Lower    : {lower:.2f}")
    print(f"Upper    : {upper:.2f}")
    print(f"Outliers : {outliers}")


===== Restaurant_latitude =====
Q1       : 12.98
Q3       : 22.75
IQR      : 9.77
Lower    : -1.67
Upper    : 37.40
Outliers : 431

===== Restaurant_longitude =====
Q1       : 73.90
Q3       : 78.35
IQR      : 4.45
Lower    : 67.22
Upper    : 85.02
Outliers : 3412

===== Delivery_location_latitude =====
Q1       : 13.07
Q3       : 22.82
IQR      : 9.75
Lower    : -1.57
Upper    : 37.45
Outliers : 0

===== Delivery_location_longitude =====
Q1       : 73.94
Q3       : 78.41
IQR      : 4.47
Lower    : 67.24
Upper    : 85.10
Outliers : 3264

===== Time_taken(min) =====
Q1       : 19.00
Q3       : 32.00
IQR      : 13.00
Lower    : -0.50
Upper    : 51.50
Outliers : 247


In [70]:
invalid = (
    (df["Restaurant_latitude"].abs() > 90) |
    (df["Delivery_location_latitude"].abs() > 90) |
    (df["Restaurant_longitude"].abs() > 180) |
    (df["Delivery_location_longitude"].abs() > 180)
)

print("Invalid coordinates:", invalid.sum())

Invalid coordinates: 0


In [71]:
col = "Time_taken(min)"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df[col] < lower) | (df[col] > upper)]

print("Q1:", Q1)
print("Q3:", Q3)
print("Lower:", lower)
print("Upper:", upper)
print("Number of outliers:", len(outliers))

Q1: 19.0
Q3: 32.0
Lower: -0.5
Upper: 51.5
Number of outliers: 247


create a distance column using the haversine formula|

In [79]:
import numpy as np
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

df["distance_km"] = haversine_distance(
    df["Restaurant_latitude"],
    df["Restaurant_longitude"],
    df["Delivery_location_latitude"],
    df["Delivery_location_longitude"]
)

print(df["distance_km"].describe())

print("Distance = 0 :", (df["distance_km"] == 0).sum())
print("Distance > 100 km :", (df["distance_km"] > 100).sum())
print("Distance > 200 km :", (df["distance_km"] > 200).sum())



count    41953.000000
mean       107.060101
std       1146.118637
min          1.465067
25%          4.658001
50%          9.220368
75%         13.760880
max      19692.674606
Name: distance_km, dtype: float64
Distance = 0 : 0
Distance > 100 km : 431
Distance > 200 km : 431


extract features from Time_Order_picked

In [ ]:
df["pickup_hour"] = df["Time_Order_picked"].dt.hour
df["pickup_minute"] = df["Time_Order_picked"].dt.minute

df["is_peak_hour"] = df["pickup_hour"].isin([12, 13, 19, 20, 21]).astype(int)

df = df.drop(columns=["Time_Order_picked"])


export result

In [83]:
result = df.to_csv(CLEAN_PATH, index=False)